# 종합문화취약지수 PCA 검토
- 목적: 종합문화취약지수 하위지표 간 중복성과 차원축소 가능성 확인
- 분석대상: 문화누리대상자 추정 인구수가 존재하는 서울 100m 격자
- 분석모형: 선호반영 H3SFCA / 선호미반영 SFCA 각각 분리 검토
- 해석주의: PCA 로딩은 정책 가중치 확정값이 아니라, 하위지표 구조를 보는 보조 진단값으로 사용

In [ ]:
import pandas as pd
import numpy as np
import pathlib
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "access":
    ACCESS_PATH = BASE_PATH
elif BASE_PATH.name == "notebooks":
    ACCESS_PATH = BASE_PATH / "access"
elif BASE_PATH.name == "oracle_mnc_project":
    ACCESS_PATH = BASE_PATH / "notebooks" / "access"
else:
    PROJECT_PATH = next(
        (path for path in [BASE_PATH, *BASE_PATH.parents] if (path / "notebooks").exists()),
        BASE_PATH,
    )
    ACCESS_PATH = PROJECT_PATH / "notebooks" / "access"

INPUT_PATH = ACCESS_PATH / "OUTPUT" / "final_vulnerability_index"
OUTPUT_PATH = ACCESS_PATH / "OUTPUT" / "vulnerability_index_pca"
IMAGE_PATH = ACCESS_PATH / "IMAGE" / "vulnerability_index_pca"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

print("ACCESS_PATH:", ACCESS_PATH)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("IMAGE_PATH:", IMAGE_PATH)

## 01. 데이터 불러오기
- 최종 종합문화취약지수 테이블 2종을 불러옴
- 선호반영 H3SFCA와 선호미반영 SFCA를 분리해서 PCA 구조를 비교함
- 분석대상 여부, 하위지표, 최종취약지수를 함께 확인함

In [ ]:
pref = pd.read_csv(
    INPUT_PATH / "종합문화취약지수_선호반영_H3SFCA.csv",
    encoding="utf-8-sig"
)

no_pref = pd.read_csv(
    INPUT_PATH / "종합문화취약지수_선호미반영_SFCA.csv",
    encoding="utf-8-sig"
)

print("선호반영 데이터 구조:", pref.shape)
print("선호미반영 데이터 구조:", no_pref.shape)
print("선호반영 분석대상 격자 수:", pref["분석대상"].sum())
print("선호미반영 분석대상 격자 수:", no_pref["분석대상"].sum())

print("\n선호반영 결측치")
print(pref[["분석대상", "최종취약지수_z", "시설접근성취약점수_z", "다양성취약점수_z", "노인편의취약점수_z", "장애인친화취약점수_z"]].isna().sum())

print("\n선호미반영 결측치")
print(no_pref[["분석대상", "최종취약지수_z", "시설접근성취약점수_z", "다양성취약점수_z", "노인편의취약점수_z", "장애인친화취약점수_z"]].isna().sum())

## 02. PCA 입력지표 설계
- PCA 입력값은 종합취약지수를 구성한 4개 하위지표로 설정함
- 모든 하위지표는 값이 클수록 취약한 방향으로 통일되어 있음
- PCA는 스케일에 민감하므로 입력 직전에 다시 표준화함

$$
X_i = [Z_{facility,i}, Z_{diversity,i}, Z_{elderly,i}, Z_{disabled,i}]
$$

$$
PC_{k,i} = a_{k1}Z_{facility,i}+a_{k2}Z_{diversity,i}+a_{k3}Z_{elderly,i}+a_{k4}Z_{disabled,i}
$$

In [ ]:
indicator_cols = [
    "시설접근성취약점수_z",
    "다양성취약점수_z",
    "노인편의취약점수_z",
    "장애인친화취약점수_z"
]

id_cols = [
    "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
    "추정_인구수", "문화누리대상자_추정_인구수",
    "최종취약지수_z", "최종취약지수_백분위", "취약등급"
]

print("PCA 입력 하위지표")
for col in indicator_cols:
    print("-", col)

for name, df in [("선호반영_H3SFCA", pref), ("선호미반영_SFCA", no_pref)]:
    temp = df[df["분석대상"] == True].copy()
    print("\n", name)
    print("분석대상:", temp.shape)
    print("입력지표 결측치:")
    print(temp[indicator_cols].isna().sum())
    print("입력지표 기초통계:")
    display(temp[indicator_cols].describe().round(4))

## 03. 하위지표 상관 점검
- PCA 전 하위지표 간 단순 상관을 확인함
- 높은 상관은 정보 중복 가능성을 의미함
- 낮은 상관은 서로 다른 취약성 차원을 담고 있을 가능성을 의미함

In [ ]:
corr_list = []

for name, df in [("선호반영_H3SFCA", pref), ("선호미반영_SFCA", no_pref)]:
    temp = df[df["분석대상"] == True].copy()
    corr = temp[indicator_cols].corr()
    corr_long = corr.reset_index().melt(
        id_vars="index",
        var_name="비교지표",
        value_name="상관계수"
    ).rename(columns={"index": "기준지표"})
    corr_long["지수모형"] = name
    corr_list.append(corr_long)

    print("\n", name, "하위지표 상관행렬")
    display(corr.round(4))

corr_result = pd.concat(corr_list, ignore_index=True)
corr_result = corr_result[["지수모형", "기준지표", "비교지표", "상관계수"]]
corr_result.to_csv(OUTPUT_PATH / "pca_하위지표상관.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", OUTPUT_PATH / "pca_하위지표상관.csv")

## 04. PCA 수행
- 하위지표 4개를 표준화한 뒤 PCA를 수행함
- 설명분산비율은 각 주성분이 전체 변동을 얼마나 설명하는지 나타냄
- 성분 로딩은 각 하위지표가 주성분 축에 어느 방향으로 강하게 연결되는지 보여줌

In [ ]:
def run_pca_review(df, model_name):
    temp = df[df["분석대상"] == True].copy()
    temp = temp.dropna(subset=indicator_cols).copy()

    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(temp[indicator_cols])

    pca = PCA(n_components=len(indicator_cols), random_state=42)
    pc_scores = pca.fit_transform(x_scaled)

    pc_cols = [f"PC{i+1}" for i in range(len(indicator_cols))]

    explained = pd.DataFrame({
        "지수모형": model_name,
        "주성분": pc_cols,
        "설명분산비율": pca.explained_variance_ratio_,
        "누적설명분산비율": np.cumsum(pca.explained_variance_ratio_),
        "고유값": pca.explained_variance_
    })

    loading = pd.DataFrame(
        pca.components_.T * np.sqrt(pca.explained_variance_),
        index=indicator_cols,
        columns=pc_cols
    ).reset_index().rename(columns={"index": "하위지표"})
    loading.insert(0, "지수모형", model_name)

    component_weight = pd.DataFrame(
        pca.components_.T,
        index=indicator_cols,
        columns=pc_cols
    ).reset_index().rename(columns={"index": "하위지표"})
    component_weight.insert(0, "지수모형", model_name)

    score = temp[id_cols].copy()
    score.insert(0, "지수모형", model_name)
    for idx, col in enumerate(pc_cols):
        score[col] = pc_scores[:, idx]

    print("\n", model_name, "PCA 설명분산")
    display(explained.round(4))

    print("\n", model_name, "PCA 성분 로딩")
    display(loading.round(4))

    return explained, loading, component_weight, score

pca_outputs = []
loading_outputs = []
component_outputs = []
score_outputs = []

for name, df in [("선호반영_H3SFCA", pref), ("선호미반영_SFCA", no_pref)]:
    explained, loading, component_weight, score = run_pca_review(df, name)
    pca_outputs.append(explained)
    loading_outputs.append(loading)
    component_outputs.append(component_weight)
    score_outputs.append(score)

pca_explained = pd.concat(pca_outputs, ignore_index=True)
pca_loading = pd.concat(loading_outputs, ignore_index=True)
pca_component_weight = pd.concat(component_outputs, ignore_index=True)
pca_score = pd.concat(score_outputs, ignore_index=True)

pca_explained.to_csv(OUTPUT_PATH / "pca_설명분산.csv", index=False, encoding="utf-8-sig")
pca_loading.to_csv(OUTPUT_PATH / "pca_성분로딩.csv", index=False, encoding="utf-8-sig")
pca_component_weight.to_csv(OUTPUT_PATH / "pca_성분계수.csv", index=False, encoding="utf-8-sig")
pca_score.to_csv(OUTPUT_PATH / "pca_격자점수.csv", index=False, encoding="utf-8-sig")

print("\n저장 완료")
print(OUTPUT_PATH / "pca_설명분산.csv")
print(OUTPUT_PATH / "pca_성분로딩.csv")
print(OUTPUT_PATH / "pca_성분계수.csv")
print(OUTPUT_PATH / "pca_격자점수.csv")

## 05. PCA 결과 요약
- PC1이 높게 설명하면 하위지표들을 하나의 공통 취약성 축으로 축소할 가능성이 있음
- PC1 설명력이 낮으면 하위지표들이 서로 다른 취약성 차원을 가진다고 해석함
- 로딩 부호는 방향성 비교용이며, 절대값이 클수록 해당 PC에 강하게 기여함

In [ ]:
for model_name in pca_explained["지수모형"].unique():
    temp_exp = pca_explained[pca_explained["지수모형"] == model_name].copy()
    temp_load = pca_loading[pca_loading["지수모형"] == model_name].copy()

    pc1_ratio = temp_exp.loc[temp_exp["주성분"] == "PC1", "설명분산비율"].iloc[0]
    pc2_cum = temp_exp.loc[temp_exp["주성분"] == "PC2", "누적설명분산비율"].iloc[0]

    temp_load["PC1_abs"] = temp_load["PC1"].abs()
    temp_load["PC2_abs"] = temp_load["PC2"].abs()

    print("\n", model_name)
    print("PC1 설명분산비율:", round(pc1_ratio, 4))
    print("PC1+PC2 누적설명분산비율:", round(pc2_cum, 4))
    print("PC1 기여 상위 지표:")
    display(temp_load.sort_values("PC1_abs", ascending=False)[["하위지표", "PC1", "PC1_abs"]].round(4))
    print("PC2 기여 상위 지표:")
    display(temp_load.sort_values("PC2_abs", ascending=False)[["하위지표", "PC2", "PC2_abs"]].round(4))

## 06. PCA 시각화
- 설명분산비율 막대그래프를 저장함
- 성분 로딩 heatmap을 저장함
- 과제용 설명자료에 바로 사용할 수 있도록 이미지 파일을 분리 저장함

In [ ]:
model_order = ["선호반영_H3SFCA", "선호미반영_SFCA"]
colors = ["#ea6b2d", "#f3a33a", "#178f85", "#8b1e16"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for ax, model_name in zip(axes, model_order):
    temp = pca_explained[pca_explained["지수모형"] == model_name].copy()
    ax.bar(temp["주성분"], temp["설명분산비율"], color=colors)
    ax.plot(temp["주성분"], temp["누적설명분산비율"], color="#2d2a26", marker="o", linewidth=2)
    ax.set_ylim(0, 1.05)
    ax.set_title(model_name)
    ax.set_ylabel("설명분산비율")
    ax.grid(axis="y", alpha=0.25)

plt.savefig(IMAGE_PATH / "pca_설명분산.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

for ax, model_name in zip(axes, model_order):
    temp = pca_loading[pca_loading["지수모형"] == model_name].copy()
    matrix = temp.set_index("하위지표")[["PC1", "PC2", "PC3", "PC4"]]
    im = ax.imshow(matrix.values, cmap="RdYlBu_r", vmin=-1, vmax=1)
    ax.set_title(model_name)
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(range(matrix.shape[0]))
    ax.set_yticklabels(matrix.index)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iloc[i, j]
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="성분 로딩")
plt.savefig(IMAGE_PATH / "pca_성분로딩.png", dpi=180, bbox_inches="tight")
plt.show()

print("이미지 저장 완료")
print(IMAGE_PATH / "pca_설명분산.png")
print(IMAGE_PATH / "pca_성분로딩.png")

## 07. 저장 결과
- PCA 설명분산, 성분 로딩, 성분계수, 격자별 PC 점수를 저장함
- 최종 정책 지수는 기존 가중합 지수를 유지함
- PCA 결과는 과제용 보조 분석 및 하위지표 중복성 검토 자료로 사용함

In [ ]:
print("PCA 산출물")
for path in sorted(OUTPUT_PATH.glob("*.csv")):
    print("-", path.name)

print("\nPCA 이미지")
for path in sorted(IMAGE_PATH.glob("*.png")):
    print("-", path.name)

## 08. 시설분류별 취약점수 확장 PCA
- 시설접근성 취약점수 1개 대신 중분류별 시설 접근성 취약점수 10개를 사용함
- 접근성지수는 값이 클수록 양호하므로, 취약점수는 부호를 반대로 변환함
- 문화다양성, 노인편의, 장애인친화 취약점수와 함께 PCA를 수행함

$$
V_{i,c}=-Z(A_{i,c})
$$

$$
X_i=[V_{i,도서},V_{i,공연},...,V_{i,체육용품},Z_{diversity,i},Z_{elderly,i},Z_{disabled,i}]
$$

In [ ]:
category_order = [
    "도서", "공연", "미술", "문화체험", "영상",
    "관광지", "체육시설", "체육용품", "스포츠관람", "음악"
]

non_facility_cols = [
    "다양성취약점수_z",
    "노인편의취약점수_z",
    "장애인친화취약점수_z"
]

access_files = {
    "선호반영_H3SFCA": INPUT_PATH.parent / "h3sfca" / "h3sfca_격자_중분류_접근성.csv",
    "선호미반영_SFCA": INPUT_PATH.parent / "h3sfca" / "sfca_no_preference_격자_중분류_접근성.csv"
}

index_files = {
    "선호반영_H3SFCA": INPUT_PATH / "종합문화취약지수_선호반영_H3SFCA.csv",
    "선호미반영_SFCA": INPUT_PATH / "종합문화취약지수_선호미반영_SFCA.csv"
}

print("시설분류별 PCA 입력 파일")
for key, value in access_files.items():
    print(key, value)

print("\n중분류 목록:", category_order)
print("비시설 하위지표:", non_facility_cols)

## 09. 시설분류별 취약점수 생성
- 중분류별 접근성지수를 분석대상 격자 기준으로 z-score 표준화함
- 접근성이 낮을수록 취약하도록 부호를 반전함
- 중분류별 값은 wide 형태로 변환해 PCA 입력변수로 사용함

In [ ]:
def make_category_vulnerability(model_name):
    index_df = pd.read_csv(index_files[model_name], encoding="utf-8-sig")
    access_df = pd.read_csv(access_files[model_name], encoding="utf-8-sig")

    base_cols = [
        "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
        "추정_인구수", "문화누리대상자_추정_인구수", "분석대상",
        "최종취약지수_z", "최종취약지수_백분위", "취약등급"
    ] + non_facility_cols

    base = index_df[base_cols].copy()
    base_target = base[base["분석대상"] == True].copy()

    access_df = access_df[access_df["GRID_CD"].isin(base_target["GRID_CD"])].copy()
    access_df = access_df[access_df["중분류"].isin(category_order)].copy()

    print("\n", model_name)
    print("기본 지수 테이블:", index_df.shape)
    print("접근성 테이블:", access_df.shape)
    print("분석대상 격자 수:", base_target["GRID_CD"].nunique())
    print("중분류 수:", access_df["중분류"].nunique())
    print("격자-중분류 중복:", access_df[["GRID_CD", "중분류"]].duplicated().sum())

    stat = access_df.groupby("중분류")["접근성지수"].agg(["mean", "std", "min", "max", "count"]).reset_index()
    stat["std"] = stat["std"].replace(0, np.nan)
    access_df = access_df.merge(stat[["중분류", "mean", "std"]], on="중분류", how="left")
    access_df["시설분류취약점수"] = -((access_df["접근성지수"] - access_df["mean"]) / access_df["std"])
    access_df["시설분류취약점수"] = access_df["시설분류취약점수"].replace([np.inf, -np.inf], np.nan).fillna(0)

    wide = access_df.pivot_table(
        index="GRID_CD",
        columns="중분류",
        values="시설분류취약점수",
        aggfunc="first"
    ).reset_index()

    wide.columns.name = None
    wide = wide.rename(columns={col: f"시설분류취약_{col}" for col in category_order if col in wide.columns})

    category_cols = [f"시설분류취약_{col}" for col in category_order]
    wide[category_cols] = wide[category_cols].fillna(0)

    pca_input = base_target.merge(wide, on="GRID_CD", how="left")
    pca_input[category_cols] = pca_input[category_cols].fillna(0)
    pca_input[non_facility_cols] = pca_input[non_facility_cols].fillna(0)

    input_cols = category_cols + non_facility_cols

    print("PCA 입력 테이블:", pca_input.shape)
    print("PCA 입력 결측:", pca_input[input_cols].isna().sum().sum())
    print("시설분류별 취약점수 기초통계")
    display(pca_input[category_cols].describe().T[["mean", "std", "min", "50%", "max"]].round(4))

    return pca_input, input_cols

expanded_input_dict = {}
expanded_input_cols = {}

for model_name in ["선호반영_H3SFCA", "선호미반영_SFCA"]:
    pca_input, input_cols = make_category_vulnerability(model_name)
    expanded_input_dict[model_name] = pca_input
    expanded_input_cols[model_name] = input_cols

## 10. 확장 PCA 입력변수 상관행렬
- 시설분류별 취약점수와 비시설 취약점수 간 상관을 확인함
- 특정 시설분류끼리 강하게 묶이는지 확인함
- 상관이 낮으면 각 분류가 서로 다른 공간 취약성을 가진다고 해석함

In [ ]:
expanded_corr_list = []

for model_name, pca_input in expanded_input_dict.items():
    input_cols = expanded_input_cols[model_name]
    corr = pca_input[input_cols].corr()

    corr_long = corr.reset_index().melt(
        id_vars="index",
        var_name="비교지표",
        value_name="상관계수"
    ).rename(columns={"index": "기준지표"})
    corr_long.insert(0, "지수모형", model_name)
    expanded_corr_list.append(corr_long)

    print("\n", model_name, "확장 PCA 상관행렬 일부")
    display(corr.round(3).iloc[:8, :8])

expanded_corr = pd.concat(expanded_corr_list, ignore_index=True)
expanded_corr.to_csv(OUTPUT_PATH / "pca_분류별확장_입력변수상관.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", OUTPUT_PATH / "pca_분류별확장_입력변수상관.csv")

## 11. 시설분류별 확장 PCA 수행
- 중분류별 시설 취약점수 10개와 비시설 취약점수 3개를 함께 PCA에 투입함
- PC1~PC4 중심으로 설명분산과 성분 로딩을 해석함
- PCA 성분은 정책 가중치가 아니라 취약성 구조를 요약하는 통계적 축으로 해석함

In [ ]:
def run_expanded_pca(model_name, pca_input, input_cols):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(pca_input[input_cols])

    pca = PCA(n_components=len(input_cols), random_state=42)
    pc_scores = pca.fit_transform(x_scaled)
    pc_cols = [f"PC{i+1}" for i in range(len(input_cols))]

    explained = pd.DataFrame({
        "지수모형": model_name,
        "주성분": pc_cols,
        "설명분산비율": pca.explained_variance_ratio_,
        "누적설명분산비율": np.cumsum(pca.explained_variance_ratio_),
        "고유값": pca.explained_variance_
    })

    loading = pd.DataFrame(
        pca.components_.T * np.sqrt(pca.explained_variance_),
        index=input_cols,
        columns=pc_cols
    ).reset_index().rename(columns={"index": "입력지표"})
    loading.insert(0, "지수모형", model_name)

    score = pca_input[[
        "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
        "추정_인구수", "문화누리대상자_추정_인구수",
        "최종취약지수_z", "최종취약지수_백분위", "취약등급"
    ]].copy()
    score.insert(0, "지수모형", model_name)
    score["PC1"] = pc_scores[:, 0]
    score["PC2"] = pc_scores[:, 1]
    score["PC3"] = pc_scores[:, 2]
    score["PC4"] = pc_scores[:, 3]

    print("\n", model_name, "확장 PCA 설명분산")
    display(explained.head(8).round(4))

    print("\n", model_name, "PC1~PC3 로딩 상위")
    for pc in ["PC1", "PC2", "PC3", "PC4"]:
        temp = loading[["입력지표", pc]].copy()
        temp["abs_loading"] = temp[pc].abs()
        print(pc)
        display(temp.sort_values("abs_loading", ascending=False).head(8).round(4))

    return explained, loading, score

expanded_explained_list = []
expanded_loading_list = []
expanded_score_list = []

for model_name in ["선호반영_H3SFCA", "선호미반영_SFCA"]:
    explained, loading, score = run_expanded_pca(
        model_name,
        expanded_input_dict[model_name],
        expanded_input_cols[model_name]
    )
    expanded_explained_list.append(explained)
    expanded_loading_list.append(loading)
    expanded_score_list.append(score)

expanded_explained = pd.concat(expanded_explained_list, ignore_index=True)
expanded_loading = pd.concat(expanded_loading_list, ignore_index=True)
expanded_score = pd.concat(expanded_score_list, ignore_index=True)

expanded_explained.to_csv(OUTPUT_PATH / "pca_분류별확장_설명분산.csv", index=False, encoding="utf-8-sig")
expanded_loading.to_csv(OUTPUT_PATH / "pca_분류별확장_성분로딩.csv", index=False, encoding="utf-8-sig")
expanded_score.to_csv(OUTPUT_PATH / "pca_분류별확장_PC점수.csv", index=False, encoding="utf-8-sig")

print("\n저장 완료")
print(OUTPUT_PATH / "pca_분류별확장_설명분산.csv")
print(OUTPUT_PATH / "pca_분류별확장_성분로딩.csv")
print(OUTPUT_PATH / "pca_분류별확장_PC점수.csv")

## 12. 시설분류별 확장 PCA 시각화
- 설명분산비율 그래프를 저장함
- PC1~PC4 성분 로딩 heatmap을 저장함
- 입력변수 상관행렬 heatmap을 저장함

In [ ]:
import matplotlib
matplotlib.use("Agg")

label_map_expanded = {
    "시설분류취약_도서": "도서",
    "시설분류취약_공연": "공연",
    "시설분류취약_미술": "미술",
    "시설분류취약_문화체험": "문화체험",
    "시설분류취약_영상": "영상",
    "시설분류취약_관광지": "관광지",
    "시설분류취약_체육시설": "체육시설",
    "시설분류취약_체육용품": "체육용품",
    "시설분류취약_스포츠관람": "스포츠관람",
    "시설분류취약_음악": "음악",
    "다양성취약점수_z": "문화다양성",
    "노인편의취약점수_z": "노인편의",
    "장애인친화취약점수_z": "장애인친화"
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

for ax, model_name in zip(axes, ["선호반영_H3SFCA", "선호미반영_SFCA"]):
    temp = expanded_explained[expanded_explained["지수모형"] == model_name].head(8).copy()
    ax.bar(temp["주성분"], temp["설명분산비율"], color="#ea6b2d")
    ax.plot(temp["주성분"], temp["누적설명분산비율"], color="#222222", marker="o", linewidth=2)
    ax.set_ylim(0, 1.0)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_ylabel("설명분산비율")
    ax.grid(axis="y", alpha=0.25)

plt.savefig(IMAGE_PATH / "pca_분류별확장_설명분산.png", dpi=220, bbox_inches="tight")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 7), constrained_layout=True)

for ax, model_name in zip(axes, ["선호반영_H3SFCA", "선호미반영_SFCA"]):
    temp = expanded_loading[expanded_loading["지수모형"] == model_name].copy()
    matrix = temp.set_index("입력지표")[["PC1", "PC2", "PC3", "PC4"]]
    matrix = matrix.loc[[col for col in label_map_expanded.keys() if col in matrix.index]]
    matrix.index = [label_map_expanded[x] for x in matrix.index]

    im = ax.imshow(matrix.values, cmap="RdYlBu_r", vmin=-1, vmax=1)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_xticklabels(matrix.columns, fontsize=10)
    ax.set_yticklabels(matrix.index, fontsize=9)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iloc[i, j]
            color = "white" if abs(value) > 0.55 else "#222222"
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8, color=color, fontweight="bold")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="성분 로딩")
plt.savefig(IMAGE_PATH / "pca_분류별확장_성분로딩.png", dpi=220, bbox_inches="tight")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), constrained_layout=True)

for ax, model_name in zip(axes, ["선호반영_H3SFCA", "선호미반영_SFCA"]):
    temp = expanded_corr[expanded_corr["지수모형"] == model_name].copy()
    matrix = temp.pivot(index="기준지표", columns="비교지표", values="상관계수")
    ordered_cols = [col for col in label_map_expanded.keys() if col in matrix.index]
    matrix = matrix.loc[ordered_cols, ordered_cols]
    matrix.index = [label_map_expanded[x] for x in matrix.index]
    matrix.columns = [label_map_expanded[x] for x in matrix.columns]

    im = ax.imshow(matrix.values, cmap="RdYlBu_r", vmin=-1, vmax=1)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_xticklabels(matrix.columns, fontsize=8, rotation=45, ha="right")
    ax.set_yticklabels(matrix.index, fontsize=8)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="Pearson 상관계수")
plt.savefig(IMAGE_PATH / "pca_분류별확장_입력변수상관행렬.png", dpi=220, bbox_inches="tight")
plt.close()

print("이미지 저장 완료")
print(IMAGE_PATH / "pca_분류별확장_설명분산.png")
print(IMAGE_PATH / "pca_분류별확장_성분로딩.png")
print(IMAGE_PATH / "pca_분류별확장_입력변수상관행렬.png")

## 13. 시설분류별 확장 PCA 해석 요약
- 시설접근성을 하나로 합치기 전의 분류별 취약성 구조를 확인함
- PC1은 여러 문화시설 분류가 동시에 부족한 공통 취약성 축으로 해석함
- PC2~PC4는 특정 시설분류 또는 특정 편의서비스가 분리되는 보조 축으로 해석함

In [ ]:
for model_name in ["선호반영_H3SFCA", "선호미반영_SFCA"]:
    temp_exp = expanded_explained[expanded_explained["지수모형"] == model_name].copy()
    temp_load = expanded_loading[expanded_loading["지수모형"] == model_name].copy()

    print("\n", model_name)
    print("PC1 설명분산비율:", round(temp_exp.loc[temp_exp["주성분"] == "PC1", "설명분산비율"].iloc[0], 4))
    print("PC1+PC2 누적설명분산비율:", round(temp_exp.loc[temp_exp["주성분"] == "PC2", "누적설명분산비율"].iloc[0], 4))
    print("PC1+PC2+PC3 누적설명분산비율:", round(temp_exp.loc[temp_exp["주성분"] == "PC3", "누적설명분산비율"].iloc[0], 4))
    print("PC1+PC2+PC3+PC4 누적설명분산비율:", round(temp_exp.loc[temp_exp["주성분"] == "PC4", "누적설명분산비율"].iloc[0], 4))

    for pc in ["PC1", "PC2", "PC3", "PC4"]:
        view = temp_load[["입력지표", pc]].copy()
        view["abs_loading"] = view[pc].abs()
        view["입력지표"] = view["입력지표"].replace(label_map_expanded)
        print("\n", pc, "로딩 상위 5개")
        display(view.sort_values("abs_loading", ascending=False).head(5).round(4))